In [35]:
import pandas as pd
import xml.etree.ElementTree as ET
import os
import re
import zipfile
import glob
import shutil
import numpy as np
import hashlib
import re
from typing import Dict, List, Tuple, Set
from tqdm.auto import tqdm
from code_compare_utils import *
from utils import parse_ape_xml, read_xml_text_safe
from functools import lru_cache
import warnings

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

path = "FOLDER_APE/MATCH/"

In [36]:


cache_bytes: Dict[str, bytes] = {}
cache_maps: Dict[str, Dict[str, str]] = {}
cache_signature: Dict[str, str] = {}

def _load_bytes(filename: str) -> bytes:
    if filename not in cache_bytes:
        with open(os.path.join(path, filename), "rb") as f:
            cache_bytes[filename] = f.read()
    return cache_bytes[filename]

def _decode_xml(raw: bytes) -> str:
    if not raw:
        return ""
    if raw.startswith(b"\xef\xbb\xbf"):
        return raw[3:].decode("utf-8")
    if raw.startswith((b"\xff\xfe", b"\xfe\xff")):
        return raw[2:].decode("utf-16")
    try:
        return raw.decode("utf-8")
    except UnicodeDecodeError:
        return raw.decode("latin-1")

def _load_map(filename: str) -> Dict[str, str]:
    if filename not in cache_maps:
        raw = _load_bytes(filename)
        content = re.sub(r"^.*?<\?xml", "<?xml", _decode_xml(raw), flags=re.DOTALL)
        cache_maps[filename] = xml_to_map(content, strip_ns=True)
    return cache_maps[filename]

def _to_float(x):
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return None
    s = str(x).strip().replace(",", ".")
    try:
        return float(s)
    except:
        return None

def _parse_date(s):
    if not s:
        return pd.NaT
    dt = pd.to_datetime(s, errors="coerce", dayfirst=True)
    if pd.isna(dt):
        dt = pd.to_datetime(s, errors="coerce")
    return dt

def _load_ape_df(file_list: list[str]) -> pd.DataFrame:
    rows = []
    for fname in file_list:
        try:
            xml_text = _decode_xml(_load_bytes(fname))
            p = parse_ape_xml(xml_text)
            servizi = set(p.get("servizi_presenti") or [])
            indirizzo = p.get("indirizzo")
            civico = p.get("civico")
            indirizzo_full = " ".join(filter(None, [indirizzo, civico])) or None
            unita = p.get("subalterno") or p.get("subA") or p.get("subDA") or os.path.splitext(fname)[0]
            
            imp_risc = p.get('impianti', {}).get('Riscaldamento', {})
            imp_acs = p.get('impianti', {}).get('ACS', {})
            imp_raf = p.get('impianti', {}).get('Raffrescamento', {})
            
            # Estrai il primo intervento raccomandato
            interventi = p.get('interventi', [])
            intervento_desc = interventi[0] if interventi else None
            
            rows.append({
                "file": fname,
                "unita": str(unita),
                "indirizzo": indirizzo_full,
                "data_emissione": _parse_date(p.get("data_emissione")),
                "classe": p.get("classe_energetica"),
                "epglnren": _to_float(p.get("epglnren")),
                "epglren": _to_float(p.get("epglren")),
                "co2": _to_float(p.get("emissioni_co2")),
                "superficie": _to_float(p.get("superficie")),
                "vettore": p.get("vettore_energetico"),
                "serv_risc": "Riscaldamento" in servizi or "climatizzazioneInvernale" in servizi,
                "serv_acs": "ACS" in servizi or "produzioneAcquaCaldaSanitaria" in servizi,
                "serv_raf": "Raffrescamento" in servizi or "climatizzazioneEstiva" in servizi,
                "serv_vent": "Ventilazione" in servizi or "ventilazioneMeccanica" in servizi,
                "serv_illu": "Illuminazione" in servizi or "illuminazione" in servizi,
                "serv_trasp": "Ascensori/Trasporto" in servizi or "trasportoPersoneCose" in servizi,
                "destinazione_uso_cod": p.get("destinazione_uso_cod"),
                "oggetto_attestato_cod": p.get("oggetto_attestato_cod"),
                "tipologia_edilizia_cod": p.get("tipologia_edilizia_cod"),
                "piano": _to_float(p.get("piano")),
                "codice_catastale": p.get("codice_catastale"),
                "sezione": p.get("sezione"),
                "foglio": p.get("foglio"),
                "particella": p.get("particella"),
                "subA": p.get("subA"),
                "subDA": p.get("subDA"),
                "subalterno": p.get("subalterno"),
                "anno_costruzione": p.get("anno_costruzione"),
                "coordinate": (p.get("lat"), p.get("lon")),
                
                "comune": p.get("comune"),
                "zona_climatica": p.get("zona_climatica"),

                # Dati miglioramento
                "intervento_desc": intervento_desc,
                "intervento_classe_target": p.get("classe_target"),
                "intervento_payback": _to_float(p.get("tempo_ritorno")),

                # Dati impianto Riscaldamento
                "imp_risc_anno": imp_risc.get('anno'),
                "imp_risc_desc": imp_risc.get('descrizione'),
                "imp_risc_tipo": imp_risc.get('tecnologia'),
                "imp_risc_epnren": _to_float(imp_risc.get('epnren')),

                # Dati impianto ACS
                "imp_acs_anno": imp_acs.get('anno'),
                "imp_acs_desc": imp_acs.get('descrizione'),
                "imp_acs_tipo": imp_acs.get('tecnologia'),
                "imp_acs_epnren": _to_float(imp_acs.get('epnren')),

                # Dati impianto Raffrescamento
                "imp_raf_anno": imp_raf.get('anno'),
                "imp_raf_desc": imp_raf.get('descrizione'),
                "imp_raf_tipo": imp_raf.get('tecnologia'),
                "imp_raf_epnren": _to_float(imp_raf.get('epnren')),
            })
        except Exception as e:
            print(f"Errore parsing {fname}: {e}")
    df = pd.DataFrame(rows)
    if not df.empty:
        df = df.sort_values(["unita","file"]).reset_index(drop=True)
    return df

In [37]:
def _load_ape_df(file_list: list[str]) -> pd.DataFrame:
    rows = []
    for fname in file_list:
        try:
            xml_text = _decode_xml(_load_bytes(fname))
            if not xml_text:
                continue
            
            p = parse_ape_xml(xml_text) # Usa il nuovo parser
            display(p)
            # MODIFICATO: Logica servizi basata sul nuovo parser
            servizi = set(p.get("servizi_presenti") or [])
            
            indirizzo = p.get("indirizzo")
            civico = p.get("civico")
            indirizzo_full = " ".join(filter(None, [indirizzo, civico])) or None
            unita = p.get("subalterno") or p.get("subA") or p.get("subDA") or os.path.splitext(os.path.basename(fname))[0]
            
            imp_risc = p.get('impianti', {}).get('Riscaldamento', {})
            imp_acs = p.get('impianti', {}).get('ACS', {})
            imp_raf = p.get('impianti', {}).get('Raffrescamento', {})
            
            interventi = p.get('interventi', [])
            intervento_desc = interventi[0] if interventi else None
            
            rows.append({
                "file": os.path.basename(fname), # Salva solo il nome del file
                "unita": str(unita),
                "indirizzo": indirizzo_full,
                "data_emissione": _parse_date(p.get("data_emissione")),
                "classe": p.get("classe_energetica"),
                "epglnren": _to_float(p.get("epglnren")),
                "epglren": _to_float(p.get("epglren")),
                "co2": _to_float(p.get("emissioni_co2")),
                "superficie": _to_float(p.get("superficie")),
                
                # MODIFICATO: Usa il nuovo campo 'vettore_energetico_principale'
                "vettore": p.get("vettore_energetico_principale"), 
                
                # MODIFICATO: Semplificata la logica dei servizi
                "serv_risc": "Riscaldamento" in servizi,
                "serv_acs": "ACS" in servizi,
                "serv_raf": "Raffrescamento" in servizi,
                "serv_vent": "Ventilazione" in servizi,
                "serv_illu": "Illuminazione" in servizi,
                "serv_trasp": "Ascensori/Trasporto" in servizi,
                
                "destinazione_uso_cod": p.get("destinazione_uso_cod"),
                "oggetto_attestato_cod": p.get("oggetto_attestato_cod"),
                "tipologia_edilizia_cod": p.get("tipologia_edilizia_cod"),
                "piano": p.get("piano"), # Lascia come stringa, verrà pulito dopo
                "codice_catastale": p.get("codice_catastale"),
                "sezione": p.get("sezione"),
                "foglio": p.get("foglio"),
                "particella": p.get("particella"),
                "subA": p.get("subA"),
                "subDA": p.get("subDA"),
                "subalterno": p.get("subalterno"),
                "anno_costruzione": p.get("anno_costruzione"),
                "coordinate": (p.get("lat"), p.get("lon")),
                
                "comune": p.get("comune"),
                "zona_climatica": p.get("zona_climatica"),

                # Dati miglioramento (già corretti dal parser)
                "intervento_desc": intervento_desc,
                "intervento_classe_target": p.get("classe_target"),
                "intervento_payback": _to_float(p.get("tempo_ritorno")),
                
                # NUOVO: Dati Involucro
                "qualita_invernale": p.get("qualita_invernale"),
                "qualita_estiva": p.get("qualita_estiva"),
                
                # NUOVO: Fonti Rinnovabili
                "fonti_rinnovabili": p.get("fonti_rinnovabili"),

                # Dati impianto (già corretti dal parser)
                "imp_risc_anno": imp_risc.get('anno'),
                "imp_risc_desc": imp_risc.get('descrizione'),
                "imp_risc_tipo": imp_risc.get('tecnologia'),
                "imp_risc_epnren": _to_float(imp_risc.get('epnren')),

                "imp_acs_anno": imp_acs.get('anno'),
                "imp_acs_desc": imp_acs.get('descrizione'),
                "imp_acs_tipo": imp_acs.get('tecnologia'),
                "imp_acs_epnren": _to_float(imp_acs.get('epnren')),

                "imp_raf_anno": imp_raf.get('anno'),
                "imp_raf_desc": imp_raf.get('descrizione'),
                "imp_raf_tipo": imp_raf.get('tecnologia'),
                "imp_raf_epnren": _to_float(imp_raf.get('epnren')),
            })
        except Exception as e:
            print(f"Errore parsing {fname}: {e}")
    df = pd.DataFrame(rows)
    if not df.empty:
        df = df.sort_values(["unita","file"]).reset_index(drop=True)
    return df

In [38]:
def create_and_save_ape_table(df: pd.DataFrame, filepath_html: str, transpose: bool = True) -> bool:
    """
    Generates a Plotly table (horizontal or vertical) and saves it as HTML and PNG.
    - transpose=True: Attributes as rows, APEs as columns (vertical).
    - transpose=False: APEs as rows, Attributes as columns (horizontal).
    """
    
    # --- MAPPE Colori e Nomi ---
    CLASSE_COLORS = {
        'A4': '#00A651', 'A3': '#53B839', 'A2': '#A5C639', 'A1': '#F3E500',
        'B': '#F8B20C', 'C': '#F58021', 'D': '#F26723', 'E': '#ED3C24',
        'F': '#ED2E24', 'G': '#ED1C24', None: '#B0B0B0', 'N/D': '#B0B0B0'
    }
    
    # NUOVO: Colori per Qualità Involucro (Slide 6)
    QUALITA_COLORS = {
        'Sorridente': '#28a745', # Verde
        'Basita/o': '#ffc107',   # Giallo
        'Triste': '#dc3545',     # Rosso
        None: '#ffffff', 'N/D': '#ffffff'
    }
    
    # NUOVO: Colori per Fonti Rinnovabili
    RINNOVABILI_COLORS = {
        'Sì': '#28a745', # Verde
        'No': '#ffffff', # Bianco (neutro)
        None: '#ffffff', 'N/D': '#ffffff'
    }

    # MODIFICATO: Rimossa VETTORE_MAP (non più necessaria)
    
    DESTINAZIONE_MAP = {
        '0': 'Residenziale', '1': 'Residenziale', '2': 'Uffici',
        '3': 'Ospedali/Cliniche', '4': 'Att. Ricreative', '5': 'Commerciale',
        '6': 'Sportive', '7': 'Scolastiche', '8': 'Industriale', None: ''
    }
    OGGETTO_MAP = {
        '0': 'Non Spec.', '1': 'Unità Singola', '2': 'Intero Edificio',
        '3': 'Gruppo Unità', None: ''
    }
    TIPOLOGIA_MAP = {
        '1': 'Ed. Isolato', '3': 'Ed. Plurifam.',
        '9': 'App. Condominio', None: ''
    }
    # -----------------------------
    
    if df.empty:
        print(f"Skipping empty DataFrame for {os.path.basename(filepath_html)}")
        return False

    filepath_png = filepath_html.replace(".html", ".png")
    orientation = "Verticale" if transpose else "Orizzontale"
    print(f"Generating {orientation.lower()} table for {os.path.basename(filepath_html)}")

    df_display = df.sort_values("data_emissione", ascending=False).copy()

    # --- Common Data Preparation ---
    catasto_parts = []
    first_row = df_display.iloc[0]
    if pd.notna(first_row.get("codice_catastale")): catasto_parts.append(f"Comune Catastale: {first_row['codice_catastale']}")
    if pd.notna(first_row.get("foglio")): catasto_parts.append(f"Foglio: {first_row['foglio']}")
    if pd.notna(first_row.get("particella")): catasto_parts.append(f"Particella: {first_row['particella']}")
    # sub_parts = [s for s in [first_row.get("subA"), first_row.get("subDA"), first_row.get("subalterno")] if pd.notna(s)]
    # if sub_parts: catasto_parts.append(f"Sub: {','.join(map(str, set(sub_parts)))}")
    catasto_subtitle = " | ".join(catasto_parts) if catasto_parts else "Dati Catastali non disponibili"

    # Servizi Attivi (Questa logica funziona ancora bene con i booleani di _load_ape_df)
    servizi_cols = {
        "serv_risc": "Risc.", "serv_acs": "ACS", "serv_raf": "Raff.",
        "serv_vent": "Vent.", "serv_illu": "Illum.", "serv_trasp": "Trasp."
    }
    def get_active_services(row):
        active = [label for col, label in servizi_cols.items() if row.get(col) is True]
        return ", ".join(active) if active else "Nessuno"
    df_display['servizi_attivi'] = df_display.apply(get_active_services, axis=1)

    # Mapped columns
    for col in ['destinazione_uso_cod', 'oggetto_attestato_cod', 'tipologia_edilizia_cod']:
        if col not in df_display.columns:
            df_display[col] = None
    dest_str = df_display['destinazione_uso_cod'].astype(str).map(DESTINAZIONE_MAP).fillna('')
    obj_str = df_display['oggetto_attestato_cod'].astype(str).map(OGGETTO_MAP).fillna('')
    tipo_str = df_display['tipologia_edilizia_cod'].astype(str).map(TIPOLOGIA_MAP).fillna('')
    df_display['descrizione_immobile'] = (dest_str + ' / ' + tipo_str + ' / ' + obj_str).str.replace(r'(^\s*/\s*|\s*/\s*$)', '', regex=True).str.replace(r'\s*/\s*/\s*', ' / ', regex=True).str.strip()
    
    # MODIFICATO: Rimuovi mapping del vettore, usa fillna. Il parser fornisce già la stringa.
    df_display['vettore'] = df_display['vettore'].fillna('Sconosciuto')

    # Formatting
    df_display['data_emissione'] = pd.to_datetime(df_display['data_emissione']).dt.strftime('%Y-%m-%d')
    df_display['epglnren'] = pd.to_numeric(df_display['epglnren'], errors='coerce').round(1)
    df_display['epglren'] = pd.to_numeric(df_display['epglren'], errors='coerce').round(1)
    df_display['superficie'] = pd.to_numeric(df_display['superficie'], errors='coerce').round(1)
    df_display['co2'] = pd.to_numeric(df_display['co2'], errors='coerce').round(1)
    df_display.loc[:, 'piano'] = df_display['piano'].astype(str).str.replace(r'\.0$', '', regex=True).replace('nan', '')
    df_display.loc[:, 'anno_costruzione'] = pd.to_numeric(df_display['anno_costruzione'], errors='coerce').astype(str).str.replace(r'\.0$', '', regex=True).replace('nan', '')
    # NUOVO: Formattazione campi
    df_display.loc[:, 'intervento_payback'] = pd.to_numeric(df_display['intervento_payback'], errors='coerce').round(1)
    df_display.loc[:, 'imp_risc_anno'] = df_display['imp_risc_anno'].astype(str).str.replace(r'\.0$', '', regex=True).replace('nan', '')
    df_display.loc[:, 'imp_acs_anno'] = df_display['imp_acs_anno'].astype(str).str.replace(r'\.0$', '', regex=True).replace('nan', '')
    # -----------------------------

    # --- Define columns and rename map ---
    # MODIFICATO: Aggiunte le nuove colonne da mostrare
    cols_to_display = [ 
        "file", "data_emissione", "classe", 
        "epglnren", "epglren", "superficie", 
        "qualita_invernale", "qualita_estiva", # NUOVO
        "vettore", "fonti_rinnovabili", # NUOVO
        "servizi_attivi", 
        "imp_risc_tipo", "imp_risc_anno", # NUOVO
        "imp_acs_tipo", "imp_acs_anno", # NUOVO
        "intervento_classe_target", "intervento_payback", # NUOVO
        "piano", "anno_costruzione", "descrizione_immobile", "co2"
    ]
    cols_existing = [col for col in cols_to_display if col in df_display.columns]

    # MODIFICATO: Aggiunti nomi per le nuove colonne
    rename_map = { 
        'file': 'File APE', 'data_emissione': 'Data Emissione', 'classe': 'Classe',
        'descrizione_immobile': 'Descrizione Immobile', 'epglnren': 'EPgl,nren (kWh/m²a)',
        'epglren': 'EPgl,ren (kWh/m²a)', 'superficie': 'Superficie (m²)',
        'piano': 'Piano', 'anno_costruzione': 'Anno Costruzione',
        'vettore': 'Vettore Principale', 'co2': 'CO₂ (kg/m²a)',
        'servizi_attivi': 'Servizi Attivi', 'unita': 'Unità Imm.',
        # NUOVI NOMI
        'qualita_invernale': 'Qualità Inv.',
        'qualita_estiva': 'Qualità Est.',
        'fonti_rinnovabili': 'Rinnovabili',
        'imp_risc_tipo': 'Tecn. Risc.',
        'imp_risc_anno': 'Anno Risc.',
        'imp_acs_tipo': 'Tecn. ACS',
        'imp_acs_anno': 'Anno ACS',
        'intervento_classe_target': 'Classe Target',
        'intervento_payback': 'Payback (Anni)'
    }
    readable_cols = [rename_map.get(col, col) for col in cols_existing]
    # ------------------------------------

    if transpose:
        # --- Vertical Table Logic ---
        if 'file' not in df_display.columns:
            print(f"Errore: Colonna 'file' non trovata per {os.path.basename(filepath_html)}")
            return False
        
        cols_existing_no_file = [c for c in cols_existing if c != 'file']
        
        df_processed = df_display.set_index('file')[cols_existing_no_file].T.reset_index()
        if 'index' not in df_processed.columns:
             print(f"Errore: Colonna 'index' non trovata dopo reset_index per {os.path.basename(filepath_html)}")
             return False
        df_processed.rename(columns={'index': 'Attributo'}, inplace=True)
        if 'Attributo' not in df_processed.columns:
             print(f"Errore: Rinomina in 'Attributo' fallita per {os.path.basename(filepath_html)}")
             return False
        df_processed['Attributo'] = df_processed['Attributo'].map(rename_map).fillna(df_processed['Attributo'])

        header_values = df_processed.columns.tolist()
        cell_values = [df_processed[col].astype(str).replace('nan', 'N/D').tolist() for col in header_values]

        # MODIFICATO: Aggiunti indici per colorazione nuove righe
        try:
            classe_row_index = df_processed[df_processed['Attributo'] == 'Classe'].index[0]
        except (IndexError, KeyError):
            classe_row_index = -1
        try:
            target_row_index = df_processed[df_processed['Attributo'] == 'Classe Target'].index[0]
        except (IndexError, KeyError):
            target_row_index = -1
        try:
            qual_inv_row_index = df_processed[df_processed['Attributo'] == 'Qualità Inv.'].index[0]
        except (IndexError, KeyError):
            qual_inv_row_index = -1
        try:
            qual_est_row_index = df_processed[df_processed['Attributo'] == 'Qualità Est.'].index[0]
        except (IndexError, KeyError):
            qual_est_row_index = -1
        try:
            rinnov_row_index = df_processed[df_processed['Attributo'] == 'Rinnovabili'].index[0]
        except (IndexError, KeyError):
            rinnov_row_index = -1


        cell_colors = []
        # Colora la prima colonna (Attributi)
        cell_colors.append(['#e9ecef' for i in range(len(df_processed))]) 
        
        # Colora le colonne dei dati (ogni APE)
        for ape_file_col in header_values[1:]:
            col_colors = []
            for i, row in df_processed.iterrows():
                val = row[ape_file_col]
                # MODIFICATO: Logica di colorazione estesa
                if i == classe_row_index or i == target_row_index:
                    col_colors.append(CLASSE_COLORS.get(val, '#B0B0B0'))
                elif i == qual_inv_row_index or i == qual_est_row_index:
                    col_colors.append(QUALITA_COLORS.get(val, '#ffffff'))
                elif i == rinnov_row_index:
                    col_colors.append(RINNOVABILI_COLORS.get(val, '#ffffff'))
                else:
                    # Colore di sfondo alternato standard
                    col_colors.append('#ffffff' if i % 2 == 0 else '#f8f9fa')
            cell_colors.append(col_colors)

        fig = go.Figure(data=[go.Table(
            columnwidth = [1.5] + [1] * (len(header_values) - 1),
            header=dict(
                values=[f"<b>{h}</b>" for h in header_values],
                fill_color='#343a40', font=dict(color='white', size=12),
                align=['left'] + ['center'] * (len(header_values) - 1),
                line_color='darkslategray'
            ),
            cells=dict(
                values=cell_values, fill_color=cell_colors,
                align=['left'] + ['center'] * (len(header_values) - 1),
                font=dict(size=11), height=28, line_color='lightgrey'
            )
        )])
        # --- End Vertical Table Logic ---

    else:
        # --- Horizontal Table Logic ---
        df_processed = df_display[cols_existing].copy()
        df_processed.fillna('N/D', inplace=True) # Riempi NaN per la visualizzazione

        header_values = readable_cols 
        cell_values = [df_processed[col].astype(str).tolist() for col in cols_existing]

        # Colori di riga alternati di default
        row_colors = ['#ffffff' if i % 2 == 0 else '#f2f2f2' for i in range(len(df_processed))]

        # MODIFICATO: Logica di colorazione estesa
        fill_color_list = []
        for col_name in cols_existing:
            if col_name == 'classe' or col_name == 'intervento_classe_target':
                fill_color_list.append(df_processed[col_name].map(CLASSE_COLORS).fillna('#B0B0B0'))
            elif col_name == 'qualita_invernale' or col_name == 'qualita_estiva':
                 fill_color_list.append(df_processed[col_name].map(QUALITA_COLORS).fillna('#ffffff'))
            elif col_name == 'fonti_rinnovabili':
                fill_color_list.append(df_processed[col_name].map(RINNOVABILI_COLORS).fillna('#ffffff'))
            else:
                fill_color_list.append(row_colors) # Colore di riga standard

        fig = go.Figure(data=[go.Table(
             header=dict(
                 values=[f"<b>{h}</b>" for h in header_values],
                 fill_color='#2c7fb8', font=dict(color='white', size=14),
                 align='left', line_color='darkslategray'
             ),
             cells=dict(
                 values=cell_values,
                 fill_color=fill_color_list, 
                 align='left', font=dict(size=12),
                 height=30, line_color='lightgrey'
             )
        )])
        # --- End Horizontal Table Logic ---

    # --- Common Layout and Saving ---
    try:
        indirizzo_titolo = df['indirizzo'].iloc[0] if not df.empty else "Indirizzo non disponibile"
    except Exception:
        indirizzo_titolo = "Indirizzo non disponibile"
    fig.update_layout(
        title=f"<b>Confronto APE ({orientation}) per: {indirizzo_titolo}</b><br><sup>{catasto_subtitle}</sup>",
        margin=dict(l=15, r=15, t=70, b=15)
    )

    try:
        pio.show(fig)
        # pio.write_html(fig, filepath_html, auto_open=False)
        # Aumentata la larghezza di default per accomodare più colonne
        # pio.write_image(fig, filepath_png, engine='kaleido', format='png', width=1800, height=800, scale=2)
        return True
    except Exception as e:
        print(f"Errore durante il salvataggio dei grafici ({orientation.lower()}) per {os.path.basename(filepath_html)}: {e}")
        return False

In [39]:
file_list = [os.path.basename(f) for f in glob.glob(os.path.join(path, "*.xml"))]
file_list = file_list[:3]
df_ape = _load_ape_df(file_list)
create_and_save_ape_table(df_ape, "APE_summary.html", transpose=True)

{'indirizzo': 'via Filedelfia 237',
 'civico': None,
 'comune': 'Torino',
 'zona_climatica': 'E',
 'anno_costruzione': '1950',
 'lat': '45.0461',
 'lon': '7.6327',
 'superficie': '23.75',
 'data_emissione': '2021-10-27',
 'destinazione_uso_cod': '0',
 'oggetto_attestato_cod': '1',
 'tipologia_edilizia_cod': '9',
 'piano': '6',
 'classe_energetica': 'G',
 'epglnren': '551.65',
 'classe_energetica_rif': 'D',
 'epglnren_rif': '142.42',
 'epglren': '132.96',
 'emissioni_co2': '130.13',
 'servizi_presenti': ['Riscaldamento', 'ACS'],
 'vettore_energetico_principale': 'Energia elettrica da rete',
 'consumi_calcolati_kwh': {'Energia elettrica da rete': 6719.0},
 'codice_catastale': 'L219',
 'sezione': None,
 'foglio': '1387',
 'particella': '35',
 'subA': '57',
 'subDA': '57',
 'subalterno': None,
 'qualita_invernale_cod': '2',
 'qualita_invernale': 'Triste',
 'qualita_estiva_cod': '2',
 'qualita_estiva': 'Triste',
 'impianti': {'Riscaldamento': {'tecnologia': 'Caldaia elettrica',
   'anno': '

{'indirizzo': 'Via livorno',
 'civico': '45',
 'comune': 'Torino',
 'zona_climatica': 'E',
 'anno_costruzione': '2002',
 'lat': '45.086973',
 'lon': '7.669217',
 'superficie': '573.19',
 'data_emissione': '2020-06-19',
 'destinazione_uso_cod': '1',
 'oggetto_attestato_cod': '1',
 'tipologia_edilizia_cod': '6',
 'piano': '0',
 'classe_energetica': 'D',
 'epglnren': '503.62',
 'classe_energetica_rif': 'B',
 'epglnren_rif': '353.29',
 'epglren': '73.96',
 'emissioni_co2': '105.07',
 'servizi_presenti': [],
 'vettore_energetico_principale': 'Gas naturale',
 'consumi_calcolati_kwh': {'Energia elettrica da rete': 90196.0,
  'Gas naturale': 112997.92},
 'codice_catastale': 'L219',
 'sezione': None,
 'foglio': '1153',
 'particella': '130',
 'subA': '162',
 'subDA': '162',
 'subalterno': None,
 'qualita_invernale_cod': '2',
 'qualita_invernale': 'Triste',
 'qualita_estiva_cod': '2',
 'qualita_estiva': 'Triste',
 'impianti': {'Riscaldamento': {'tecnologia': 'Caldaia standard',
   'anno': '2002',

{'indirizzo': 'via Filedelfia 237',
 'civico': None,
 'comune': 'Torino',
 'zona_climatica': 'E',
 'anno_costruzione': '1950',
 'lat': '45.0461',
 'lon': '7.6327',
 'superficie': '57.77',
 'data_emissione': '2021-10-27',
 'destinazione_uso_cod': '0',
 'oggetto_attestato_cod': '1',
 'tipologia_edilizia_cod': '9',
 'piano': '5',
 'classe_energetica': 'D',
 'epglnren': '94.16',
 'classe_energetica_rif': 'C',
 'epglnren_rif': '64.26',
 'epglren': '0.64',
 'emissioni_co2': '18.93',
 'servizi_presenti': ['Riscaldamento', 'ACS'],
 'vettore_energetico_principale': 'Teleriscaldamento',
 'consumi_calcolati_kwh': {'Energia elettrica da rete': 79.0,
  'Teleriscaldamento': 3523.0},
 'codice_catastale': 'L219',
 'sezione': None,
 'foglio': '1387',
 'particella': '35',
 'subA': '53',
 'subDA': '53',
 'subalterno': None,
 'qualita_invernale_cod': '2',
 'qualita_invernale': 'Triste',
 'qualita_estiva_cod': '1',
 'qualita_estiva': 'Basita/o',
 'impianti': {'Riscaldamento': {'tecnologia': 'Teleriscaldame

Generating verticale table for APE_summary.html


True